In [ ]:
pip install numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install librosa


In [ ]:
pip install pyannote.metrics pyannote.core


In [ ]:
# make_frame_labels.py
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/mixes")
FEAT_ROOT = ROOT / "features_logmel"
SR = 48000
HOP = 512
DT = HOP / SR
N_CLASSES = 4

with open(ROOT / "dataset.json","r",encoding="utf-8") as f:
    REF = json.load(f)

# index annotations by audio_id
by_audio = {}
for a in REF["audios"]:
    by_audio[a["id"]] = {"duration": float(a["duration"]), "anns": []}
for an in REF["annotations"]:
    aid = an["audio_id"]
    by_audio[aid]["anns"].append((int(an["category_id"])-1, float(an["start_time"]), float(an["end_time"])))

def process_split(split):
    df = pd.read_csv(FEAT_ROOT / f"{split}.csv")
    y_paths = []
    for _,row in df.iterrows():
        aid = int(row["id"]) if "id" in df.columns else None
        feat_path = Path(row["feature_path"])
        M = np.load(feat_path)            # [n_mels, T]
        T = M.shape[1]
        Y = np.zeros((T, N_CLASSES), dtype=np.float32)
        if aid is None:
            # fall back: parse stem 'mix_XXXX' to audio index
            try:
                aid = int(Path(row["file_name"]).stem.split("_")[-1])
            except:
                aid = 0
        if aid in by_audio:
            for cid, s, e in by_audio[aid]["anns"]:
                i0 = int(np.floor(s / DT))
                i1 = int(np.ceil(e / DT))
                i0 = max(0, min(T, i0)); i1 = max(0, min(T, i1))
                if i1 > i0:
                    Y[i0:i1, cid] = 1.0
        y_path = feat_path.with_suffix(".y.npy")
        np.save(y_path, Y)
        y_paths.append(str(y_path))
    df["label_path"] = y_paths
    df.to_csv(FEAT_ROOT / f"{split}.csv", index=False)
    print("Wrote labels for", split)

if __name__ == "__main__":
    for s in ["train","val","test"]:
        process_split(s)


Wrote labels for train
Wrote labels for val
Wrote labels for test


In [ ]:
# train_crnn_sed.py (improved, CPU-friendly)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm

ROOT = Path("/content/drive/MyDrive/mixes")
FEAT_ROOT = ROOT / "features_logmel"
OUT_DIR = ROOT / "models"
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 6
EPOCHS = 35              # modestly longer; early stop will cap
LR = 1e-3
N_CLASSES = 4
N_MELS = 64              # must match cache_features.py
SR = 48000
HOP = 512

# Loss/imbalance
USE_FOCAL = True
FOCAL_GAMMA = 1.5
CLASS_WEIGHTS = torch.tensor([1.0, 2.0, 1.5, 2.0], dtype=torch.float32)  # upweight classes 2 & 4, tweak as needed

# Regularization
USE_LOGIT_MEDIAN = True
LOGIT_MEDIAN_K = 3  # frames

# Validation-aligned to PDF
FINAL_ROUND_TO_SEC = True

def seconds_per_frame():
    return HOP / SR

def frames_to_seconds_idx(T_frames):
    spf = seconds_per_frame()
    return np.arange(T_frames, dtype=np.float32) * spf

class FrameSet(Dataset):
    def __init__(self, split_csv):
        self.df = pd.read_csv(FEAT_ROOT / split_csv)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        fp = self.df.iloc[i]["feature_path"]
        lp = self.df.iloc[i]["label_path"]
        M = np.load(fp).astype(np.float32)   # [M,T]
        Y = np.load(lp).astype(np.float32)   # [T,C], weak/strong labels at frame-rate
        x = torch.from_numpy(M).unsqueeze(0) # [1,M,T]
        y = torch.from_numpy(Y)              # [T,C]
        return x, y

def collate_pad(batch):
    maxT = max(x.shape[-1] for x,y in batch)
    Ms, Ys = [], []
    for x,y in batch:
        T = x.shape[-1]
        if T < maxT:
            pad = maxT - T
            x = torch.nn.functional.pad(x, (0,pad))
            y = torch.nn.functional.pad(y, (0,0,0,pad))
        Ms.append(x); Ys.append(y)
    return torch.stack(Ms,0), torch.stack(Ys,0)

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=N_CLASSES):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2))
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400, dtype=torch.float32)
            z = self.cnn(dummy)
            _, C, Mp, Tp = z.shape
            self._feat_per_step = C * Mp
        self.rnn = nn.GRU(input_size=self._feat_per_step, hidden_size=128,
                          batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        # x: [B,1,M,T]
        z = self.cnn(x)  # [B,C,M’,T’]
        B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)  # [B,T’,F]
        z,_ = self.rnn(z)  # [B,T’,256]
        return self.head(z)  # [B,T’,4]

def align_time(logits, labels):
    Bt, Tt, C = logits.shape
    Tl = labels.shape[1]
    if Tt > Tl:
        logits = logits[:, :Tl]
    elif Tt < Tl:
        pad = Tl - Tt
        logits = torch.nn.functional.pad(logits, (0,0,0,pad))
    return logits

class BalancedFocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=1.5):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights  # Tensor [C] or None
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
    def forward(self, logits, targets):
        # logits: [B,T,C], targets: [B,T,C]
        bce = self.bce(logits, targets)  # [B,T,C]
        with torch.no_grad():
            p = torch.sigmoid(logits)
        focal = ((1 - torch.abs(p - targets)) ** self.gamma) * bce
        if self.class_weights is not None:
            focal = focal * self.class_weights.view(1,1,-1)
        return focal.mean()

def apply_logit_median_regularizer(logits, k=3):
    # logits: [B,T,C]
    if k <= 1: return logits
    # 1D median filter over time using unfold (CPU-friendly)
    B,T,C = logits.shape
    pad = k // 2
    x = logits.permute(0,2,1)  # [B,C,T]
    x = torch.nn.functional.pad(x, (pad,pad), mode='replicate')  # [B,C,T+2p]
    x_unf = x.unfold(dimension=2, size=k, step=1)  # [B,C,T,k]
    x_med, _ = x_unf.median(dim=-1)  # [B,C,T]
    x_med = x_med.permute(0,2,1)  # [B,T,C]
    # blend a bit to avoid over-smoothing
    return 0.5*logits + 0.5*x_med

def eventize_seconds(probs, thr_vec, hop_sec, clip_duration_sec):
    """
    Convert frame probs [T,C] to non-overlapping, nearest-second events aligned to evaluation rules.
    Returns list of events dicts: {cat, st_int, et_int, score}
    """
    T, C = probs.shape
    events = []
    # simple per-class masking
    for c in range(C):
        thr = float(thr_vec[c])
        mask = probs[:, c] >= thr
        i = 0
        while i < T:
            if mask[i]:
                j = i
                smax = probs[i, c]
                while j < T and mask[j]:
                    smax = max(smax, probs[j, c])
                    j += 1
                st = i * hop_sec
                et = j * hop_sec
                st_i = int(np.rint(st)); et_i = int(np.rint(et))
                st_i = max(0, min(st_i, int(np.floor(clip_duration_sec))))
                et_i = max(0, min(et_i, int(np.floor(clip_duration_sec))))
                if et_i > st_i:
                    events.append({"cat": c+1, "st": st_i, "et": et_i, "score": float(smax)})
                i = j
            else:
                i += 1
    if not events:
        return []
    # sort and globally de-overlap (same as submission logic, simplified)
    events.sort(key=lambda e: (e["st"], -e["score"]))
    out = []
    for ev in events:
        if not out:
            out.append(ev); continue
        last = out[-1]
        if ev["st"] >= last["et"]:
            out.append(ev)
        else:
            # overlap: prefer higher score; try to split if >=1 s overlap
            overlap_w = min(ev["et"], last["et"]) - ev["st"]
            if overlap_w >= 1:
                split_t = int(np.rint((ev["st"] + min(ev["et"], last["et"])) / 2.0))
                last["et"] = max(last["et"], split_t)
                if ev["et"] - split_t >= 1:
                    ev2 = dict(ev); ev2["st"] = split_t
                    out.append(ev2)
            else:
                if ev["score"] > last["score"]:
                    # push current after last if room
                    st_new = last["et"]
                    if ev["et"] - st_new >= 1:
                        ev2 = dict(ev); ev2["st"] = st_new
                        out.append(ev2)
                else:
                    # keep last; try to shift current
                    st_new = last["et"]
                    if ev["et"] - st_new >= 1:
                        ev2 = dict(ev); ev2["st"] = st_new
                        out.append(ev2)
    # merge touching within same cat
    merged = []
    for ev in out:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["st"] <= last["et"]:
            last["et"] = max(last["et"], ev["et"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

def temperature_scale(logits, T):
    return logits / T

def calibrate_temperature(val_loader, model, init_T=1.0):
    # Lightweight scalar T calibration minimizing BCE on val
    T = torch.tensor([init_T], dtype=torch.float32, requires_grad=True, device=DEVICE)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=20, line_search_fn='strong_wolfe')
    crit = nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad()
        loss_acc = 0.0
        n = 0
        with torch.no_grad():
            pass
        for x,y in val_loader:
            x = x.to(DEVICE); y = y.to(DEVICE)
            logits = model(x)
            logits = align_time(logits, y)
            logits_scaled = logits / T.clamp(min=0.1, max=10.0)
            loss = crit(logits_scaled, y)
            loss.backward()
            loss_acc += loss.item()
            n += 1
        return loss_acc / max(n,1)
    try:
        opt.step(closure)
    except Exception:
        pass
    return float(T.detach().cpu().item())

def train():
    tr = FrameSet("train.csv"); va = FrameSet("val.csv")
    # Lower workers for Colab CPU memory stability
    tr_dl = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=0, collate_fn=collate_pad, pin_memory=False)
    va_dl = DataLoader(va, batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate_pad, pin_memory=False)

    model = CRNN_SED().to(DEVICE)
    base_crit = BalancedFocalLoss(class_weights=CLASS_WEIGHTS.to(DEVICE) if USE_FOCAL else None,
                                  gamma=FOCAL_GAMMA)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    best_val_metric = -1.0
    best_state = None

    # Simple static thresholds for validation eventization; tune if needed
    val_thr = np.array([0.3, 0.2, 0.2, 0.15], dtype=np.float32)
    hop_sec = seconds_per_frame()

    for ep in range(1, EPOCHS+1):
        model.train(); tr_loss=0.0; ntr=0
        for x,y in tqdm(tr_dl, desc=f"train {ep}", leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)                      # [B,T’,C]
            logits = align_time(logits, y)
            if USE_LOGIT_MEDIAN:
                logits = apply_logit_median_regularizer(logits, k=LOGIT_MEDIAN_K)
            loss = base_crit(logits, y)
            loss.backward()
            opt.step()
            tr_loss += float(loss.item())*x.size(0); ntr += x.size(0)
        tr_loss /= max(ntr,1)

        # Eval
        model.eval(); vl_loss=0.0; nva=0
        # Collect a small subset for temperature calibration every few epochs
        if ep in (5, 10, 15):
            _ = calibrate_temperature(va_dl, model, init_T=1.0)
        with torch.no_grad():
            for x,y in tqdm(va_dl, desc=f"val {ep}", leave=False):
                x,y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                logits = align_time(logits, y)
                loss = nn.BCEWithLogitsLoss()(logits, y)  # report plain BCE for reference
                vl_loss += float(loss.item())*x.size(0); nva += x.size(0)
        vl_loss /= max(nva,1)

        # Event-level metric aligned to submission: 1 s rounding and non-overlap
        # We compute a simple per-class event count recall proxy on val labels if they are strong.
        ev_correct = 0; ev_total = 0
        with torch.no_grad():
            for x,y in va_dl:
                x = x.to(DEVICE); y_np = y.numpy()  # labels on CPU
                logits = model(x)
                logits = align_time(logits, y)
                probs = torch.sigmoid(logits).cpu().numpy()  # [B,T,C]
                B,T,C = probs.shape
                for b in range(B):
                    # predicted events
                    pred_events = eventize_seconds(probs[b], val_thr, hop_sec, clip_duration_sec=T*hop_sec)
                    # proxy GT events: take per-class runs of y>=0.5, then round to seconds and de-overlap
                    gt_events = eventize_seconds(y_np[b], np.array([0.5,0.5,0.5,0.5], dtype=np.float32),
                                                 hop_sec, clip_duration_sec=T*hop_sec)
                    ev_total += len(gt_events)
                    # a naive matching: count pred events that exactly match class and intersect at least 1 s
                    for ge in gt_events:
                        for pe in pred_events:
                            if ge["cat"] != pe["cat"]: continue
                            inter = max(0, min(ge["et"], pe["et"]) - max(ge["st"], pe["st"]))
                            if inter >= 1:
                                ev_correct += 1
                                break
        val_recall = (ev_correct / ev_total) if ev_total > 0 else 0.0

        print({"epoch":ep,"train_loss":tr_loss,"val_loss":vl_loss,"val_event_recall":round(val_recall,4)})

        # Early stopping on event recall
        if val_recall > best_val_metric:
            best_val_metric = val_recall
            best_state = {k: v.cpu() for k,v in model.state_dict().items()}
            torch.save(best_state, OUT_DIR / "crnn_sed.pt")

    if best_state is None:
        torch.save(model.state_dict(), OUT_DIR / "crnn1627_sed.pt")
    print("Saved:", OUT_DIR / "crnn1627_sed.pt")

if __name__ == "__main__":
    train()


{'epoch': 1, 'train_loss': 0.30153788909316065, 'val_loss': 0.5571647036075592, 'val_event_recall': 1.0}


{'epoch': 2, 'train_loss': 0.2989682278782129, 'val_loss': 0.5613724589347839, 'val_event_recall': 1.0}


{'epoch': 3, 'train_loss': 0.29892529651522637, 'val_loss': 0.5640120410919189, 'val_event_recall': 1.0}


{'epoch': 4, 'train_loss': 0.2989092871546745, 'val_loss': 0.5660652089118957, 'val_event_recall': 1.0}


{'epoch': 5, 'train_loss': 0.2989014902710915, 'val_loss': 0.5677518713474273, 'val_event_recall': 1.0}


...

{'epoch': 31, 'train_loss': 0.29888515911996366, 'val_loss': 0.5842477226257324, 'val_event_recall': 1.0}


{'epoch': 32, 'train_loss': 0.29888511762022973, 'val_loss': 0.584644238948822, 'val_event_recall': 1.0}


{'epoch': 33, 'train_loss': 0.29888510197401047, 'val_loss': 0.5850335109233856, 'val_event_recall': 1.0}


{'epoch': 34, 'train_loss': 0.29888509660959245, 'val_loss': 0.585418975353241, 'val_event_recall': 1.0}


{'epoch': 35, 'train_loss': 0.2988850785046816, 'val_loss': 0.5858036804199219, 'val_event_recall': 1.0}
Saved: /content/drive/MyDrive/mixes/models/crnn1627_sed.pt


In [ ]:
# build_submission_robust.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt # Import from scipy.signal

ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "20251006_PS12"
OUT_JSON = ROOT / "submission_1815.json"

SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Posterior thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Energy gate for vessel fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5            # median + Z*mad threshold

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def median_filter_probs(P, k=5):
    try:
        from scipy.ndimage import median_filter
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    # bandpass energy -> per-second RMS
    # use scipy.signal.iirfilter and scipy.signal.sosfiltfilt
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                            btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    # frame energy
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    # robust threshold
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    # map to seconds
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    # 60s window: ensure at least 1 vessel second per minute if energy present
    for s in range(0, total_sec, 60):
        seg = sec_flags[s:s+60]
        if seg.sum() == 0 and hot.sum() > 0:
            # pick the second with highest rms in that minute
            # approximate mapping: take frame index near center of window
            idx = min(len(rms)-1, int(np.rint(((s+30)*SR)/ENERGY_HOP)))
            sec_pick = min(total_sec-1, s + 30)
            sec_flags[sec_pick] = 1
    return sec_flags

def per_second_fusion(P, y, clip_duration):
    # P: [T,4] posterior; y: waveform
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None, 0]
    # aggregate max per second
    sec_scores = np.full((total_sec, 4), -np.inf, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])
    # voters
    voters = []
    # A: argmax
    voters.append(np.argmax(np.where(np.isfinite(sec_scores), sec_scores, -1e9), axis=1))
    # B: threshold masks
    thr = compute_adaptive_thresholds(P)
    masks = (np.where(np.isfinite(sec_scores), sec_scores, -1e9) >= thr.reshape(1,4)).astype(np.int32)
    # choose highest class among those passing, else -1
    b_choice = np.where(masks.sum(1)>0, np.argmax(sec_scores, axis=1), -1)
    voters.append(b_choice)
    # C: energy vessel fallback
    vessel_secs = energy_vessel_seconds(y, total_sec)
    c_choice = np.where(vessel_secs==1, 0, -1)  # 0-based vessel
    voters.append(c_choice)

    # majority fusion per second
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = [v[s] for v in voters if v[s] != -1]
        if not votes:
            # fallback to global argmax row if all -1
            row = sec_scores[s]
            if np.all(np.isneginf(row)):
                sec_cls[s] = None
                continue
            best = int(np.argmax(row))
            sec_cls[s] = best + 1
            continue
        # vote tally
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        if counts[winner] >= 2:
            sec_cls[s] = winner + 1
        else:
            # tie-break by posterior
            best = int(np.argmax(sec_scores[s]))
            sec_cls[s] = best + 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        P = torch.sigmoid(model(x)[0]).cpu().numpy().astype(np.float32)
    sec_cls, sec_scores = per_second_fusion(P, y, wav_duration)
    # to segments
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = sec_scores[s, c-1] if np.isfinite(sec_scores[s, c-1]) else 0.0
        s += 1
        while s < total_sec and sec_cls[s] == c:
            if np.isfinite(sec_scores[s, c-1]):
                max_sc = max(max_sc, sec_scores[s, c-1])
            s += 1
        et = s
        segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    return segs

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()

Wrote submission with 917 annotations to /content/drive/MyDrive/submission_1815.json
Counts by category: {1: 465, 2: 394, 3: 47, 4: 11}


In [ ]:
# build_submission_robust.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt # Import from scipy.signal

ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "Mock_Data_4_20251013_PS12/I4_20251013_PS12"
OUT_JSON = ROOT / "submission_1529.json"

SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Posterior thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Energy gate for vessel fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5            # median + Z*mad threshold

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def median_filter_probs(P, k=5):
    try:
        from scipy.ndimage import median_filter
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    # bandpass energy -> per-second RMS
    # use scipy.signal.iirfilter and scipy.signal.sosfiltfilt
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                            btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    # frame energy
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    # robust threshold
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    # map to seconds
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    # 60s window: ensure at least 1 vessel second per minute if energy present
    for s in range(0, total_sec, 60):
        seg = sec_flags[s:s+60]
        if seg.sum() == 0 and hot.sum() > 0:
            # pick the second with highest rms in that minute
            # approximate mapping: take frame index near center of window
            idx = min(len(rms)-1, int(np.rint(((s+30)*SR)/ENERGY_HOP)))
            sec_pick = min(total_sec-1, s + 30)
            sec_flags[sec_pick] = 1
    return sec_flags

def per_second_fusion(P, y, clip_duration):
    # P: [T,4] posterior; y: waveform
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None, 0]
    # aggregate max per second
    sec_scores = np.full((total_sec, 4), -np.inf, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])
    # voters
    voters = []
    # A: argmax
    voters.append(np.argmax(np.where(np.isfinite(sec_scores), sec_scores, -1e9), axis=1))
    # B: threshold masks
    thr = compute_adaptive_thresholds(P)
    masks = (np.where(np.isfinite(sec_scores), sec_scores, -1e9) >= thr.reshape(1,4)).astype(np.int32)
    # choose highest class among those passing, else -1
    b_choice = np.where(masks.sum(1)>0, np.argmax(sec_scores, axis=1), -1)
    voters.append(b_choice)
    # C: energy vessel fallback
    vessel_secs = energy_vessel_seconds(y, total_sec)
    c_choice = np.where(vessel_secs==1, 0, -1)  # 0-based vessel
    voters.append(c_choice)

    # majority fusion per second
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = [v[s] for v in voters if v[s] != -1]
        if not votes:
            # fallback to global argmax row if all -1
            row = sec_scores[s]
            if np.all(np.isneginf(row)):
                sec_cls[s] = None
                continue
            best = int(np.argmax(row))
            sec_cls[s] = best + 1
            continue
        # vote tally
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        if counts[winner] >= 2:
            sec_cls[s] = winner + 1
        else:
            # tie-break by posterior
            best = int(np.argmax(sec_scores[s]))
            sec_cls[s] = best + 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        P = torch.sigmoid(model(x)[0]).cpu().numpy().astype(np.float32)
    sec_cls, sec_scores = per_second_fusion(P, y, wav_duration)
    # to segments
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = sec_scores[s, c-1] if np.isfinite(sec_scores[s, c-1]) else 0.0
        s += 1
        while s < total_sec and sec_cls[s] == c:
            if np.isfinite(sec_scores[s, c-1]):
                max_sc = max(max_sc, sec_scores[s, c-1])
            s += 1
        et = s
        segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    return segs

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()

Wrote submission with 1035 annotations to /content/drive/MyDrive/submission_1529.json
Counts by category: {1: 518, 2: 413, 3: 75, 4: 29}


In [ ]:
# build_submission_ps12_gtaligned.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# ----------------------------
# Paths
# ----------------------------
ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "Mock_Data_4_20251013_PS12/I4_20251013_PS12"  # update if needed
OUT_JSON = ROOT / "submission_ps12_gtaligned.json"

# ----------------------------
# Audio/feature params
# ----------------------------
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# ----------------------------
# Post-proc and calibration
# ----------------------------
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Adaptive thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.003

# De-chatter
REASSIGN_MARGIN = 0.02  # score advantage needed to reassign 1 s flips

# Vessel continuity
GAP_FILL_SEC = 10
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

# ----------------------------
# Model
# ----------------------------
class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

# ----------------------------
# Featurization
# ----------------------------
def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

# ----------------------------
# Calibration
# ----------------------------
def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    # per-class min-max stretch
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

# ----------------------------
# Vessel energy fallback
# ----------------------------
def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

# ----------------------------
# Per-second decision and GT-style stitching
# ----------------------------
def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    # per-second max scores
    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    # voters: argmax, threshold mask, energy vessel
    thr = compute_adaptive_thresholds(P)
    # mask voter
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    # argmax voter
    argmax_choice = np.argmax(sec_scores, axis=1)
    # energy vessel voter
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    # majority fusion
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1  # 1..4

    # de-chatter: reassign isolated 1 s flips to neighbor with margin
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # vessel continuity: fill small gaps between strong vessel blocks, and extend edges
    def strong_vessel(sec): return sec_scores[sec,0] >= VESSEL_STRONG
    # identify vessel blocks
    s = 0
    vessel_blocks = []
    while s < total_sec:
        if sec_cls[s] == 1:
            st = s; s += 1
            while s < total_sec and sec_cls[s] == 1: s += 1
            vessel_blocks.append((st, s))
        else:
            s += 1
    # fill gaps <= GAP_FILL_SEC if both sides are strong vessel
    for i in range(len(vessel_blocks)-1):
        a_st, a_et = vessel_blocks[i]
        b_st, b_et = vessel_blocks[i+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (strong_vessel(a_et-1) if a_et-1 >=0 else False) and (strong_vessel(b_st) if b_st<total_sec else False):
                for k in range(a_et, b_st):
                    # only fill unassigned or weakly assigned seconds (score margin)
                    cur_c = sec_cls[k]
                    if cur_c != 1 and (cur_c is None or sec_scores[k,0] >= max(PRUNE_MIN_VESSEL, 0.5*sec_scores[k,cur_c-1])):
                        sec_cls[k] = 1

    # extend edges for strong vessel up to EDGE_EXTEND_SEC
    # left edge
    for s in range(min(EDGE_EXTEND_SEC, total_sec)):
        if sec_cls[s] is None and (total_sec>0 and sec_scores[0,0] >= VESSEL_STRONG):
            sec_cls[s] = 1
    # right edge
    for off in range(1, EDGE_EXTEND_SEC+1):
        idx = total_sec - off
        if idx >= 0 and sec_cls[idx] is None and sec_scores[-1,0] >= VESSEL_STRONG:
            sec_cls[idx] = 1

    return sec_cls, sec_scores

# ----------------------------
# Run one file
# ----------------------------
def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    # per-audio calibration
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    # build segments, prune ultra-low confidence
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1])
        s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL:
                continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER:
                continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    # merge touching same-class (defensive)
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

# ----------------------------
# IO and validation
# ----------------------------
def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

# ----------------------------
# Main
# ----------------------------
def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1218 annotations to /content/drive/MyDrive/submission_ps12_gtaligned.json
Counts by category: {1: 401, 2: 387, 3: 285, 4: 145}


In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# Paths
ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "Mock_Data_4_20251013_PS12/I4_20251013_PS12"  # update if needed
OUT_JSON = ROOT / "submission_1742.json"

# Audio/feature
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# Post-proc and calibration
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (tightened non-vessel)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004  # was 0.003

# De-chatter (stronger)
REASSIGN_MARGIN = 0.03    # was 0.02

# Vessel continuity (wider gap fill)
GAP_FILL_SEC = 12         # was 10
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # de-chatter (stronger margin)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # vessel continuity (wider gap fill and edges)
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1])
        s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1221 annotations to /content/drive/MyDrive/submission_1742.json
Counts by category: {1: 418, 2: 391, 3: 275, 4: 137}


#FINAL TRY 1

In [ ]:
# build_submission_robust.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt # Import from scipy.signal

ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT /"20251027_PS12/20251027_PS12"
OUT_JSON = ROOT / "submission_1125_finale.json"

SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Posterior thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Energy gate for vessel fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5            # median + Z*mad threshold

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def median_filter_probs(P, k=5):
    try:
        from scipy.ndimage import median_filter
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    # bandpass energy -> per-second RMS
    # use scipy.signal.iirfilter and scipy.signal.sosfiltfilt
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                            btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    # frame energy
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    # robust threshold
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    # map to seconds
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    # 60s window: ensure at least 1 vessel second per minute if energy present
    for s in range(0, total_sec, 60):
        seg = sec_flags[s:s+60]
        if seg.sum() == 0 and hot.sum() > 0:
            # pick the second with highest rms in that minute
            # approximate mapping: take frame index near center of window
            idx = min(len(rms)-1, int(np.rint(((s+30)*SR)/ENERGY_HOP)))
            sec_pick = min(total_sec-1, s + 30)
            sec_flags[sec_pick] = 1
    return sec_flags

def per_second_fusion(P, y, clip_duration):
    # P: [T,4] posterior; y: waveform
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None, 0]
    # aggregate max per second
    sec_scores = np.full((total_sec, 4), -np.inf, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])
    # voters
    voters = []
    # A: argmax
    voters.append(np.argmax(np.where(np.isfinite(sec_scores), sec_scores, -1e9), axis=1))
    # B: threshold masks
    thr = compute_adaptive_thresholds(P)
    masks = (np.where(np.isfinite(sec_scores), sec_scores, -1e9) >= thr.reshape(1,4)).astype(np.int32)
    # choose highest class among those passing, else -1
    b_choice = np.where(masks.sum(1)>0, np.argmax(sec_scores, axis=1), -1)
    voters.append(b_choice)
    # C: energy vessel fallback
    vessel_secs = energy_vessel_seconds(y, total_sec)
    c_choice = np.where(vessel_secs==1, 0, -1)  # 0-based vessel
    voters.append(c_choice)

    # majority fusion per second
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = [v[s] for v in voters if v[s] != -1]
        if not votes:
            # fallback to global argmax row if all -1
            row = sec_scores[s]
            if np.all(np.isneginf(row)):
                sec_cls[s] = None
                continue
            best = int(np.argmax(row))
            sec_cls[s] = best + 1
            continue
        # vote tally
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        if counts[winner] >= 2:
            sec_cls[s] = winner + 1
        else:
            # tie-break by posterior
            best = int(np.argmax(sec_scores[s]))
            sec_cls[s] = best + 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        P = torch.sigmoid(model(x)[0]).cpu().numpy().astype(np.float32)
    sec_cls, sec_scores = per_second_fusion(P, y, wav_duration)
    # to segments
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = sec_scores[s, c-1] if np.isfinite(sec_scores[s, c-1]) else 0.0
        s += 1
        while s < total_sec and sec_cls[s] == c:
            if np.isfinite(sec_scores[s, c-1]):
                max_sc = max(max_sc, sec_scores[s, c-1])
            s += 1
        et = s
        segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    return segs

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()

Wrote submission with 660 annotations to /content/drive/MyDrive/submission_1125_finale.json
Counts by category: {1: 355, 2: 282, 3: 21, 4: 2}


# final try 2

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# Paths
ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "20251027_PS12/20251027_PS12"  # update if needed
OUT_JSON = ROOT / "submission_1143_finale.json"

# Audio/feature
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# Post-proc and calibration
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (tightened non-vessel)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004  # was 0.003

# De-chatter (stronger)
REASSIGN_MARGIN = 0.03    # was 0.02

# Vessel continuity (wider gap fill)
GAP_FILL_SEC = 12         # was 10
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # de-chatter (stronger margin)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # vessel continuity (wider gap fill and edges)
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1])
        s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1198 annotations to /content/drive/MyDrive/submission_1143_finale.json
Counts by category: {1: 460, 2: 338, 4: 144, 3: 256}


# final try 3

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# ----------------------------
# Paths
# ----------------------------
ROOT = Path("/content/drive/MyDrive")
WAV_DIR = ROOT / "20251027_PS12/20251027_PS12"  # set to your target mock dataset folder
MODEL_PATH = ROOT / "mixes/models/crnn_sed_finetuned_both_smallwin.pt"
OUT_JSON = ROOT / "submission_1159_finale.json"

# ----------------------------
# Audio/feature
# ----------------------------
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# ----------------------------
# Post-proc and calibration
# ----------------------------
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Adaptive thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (strict)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004

# De-chatter
REASSIGN_MARGIN = 0.03  # stronger

# Vessel continuity
GAP_FILL_SEC = 12
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

# ----------------------------
# Model
# ----------------------------
class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            self._feat_per_step = C*Mp
        self.rnn = nn.GRU(input_size=self._feat_per_step, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x)
        B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

# ----------------------------
# Calibration
# ----------------------------
def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

# ----------------------------
# Energy fallback
# ----------------------------
def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

# ----------------------------
# Per-second fusion and stitching
# ----------------------------
def frames_to_seconds(i): return i * (HOP / SR)

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # De-chatter (remove isolated 1s flips)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # Vessel continuity: fill small gaps and edge extend
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

# ----------------------------
# Inference for one file
# ----------------------------
def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def run_file(model, wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    # per-audio calibration
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    # build segments + prune
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1]); s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class defensively
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

# ----------------------------
# IO and validation
# ----------------------------
def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

# ----------------------------
# Main
# ----------------------------
def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    model = CRNN_SED()
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    model.eval()

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(model, a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1215 annotations to /content/drive/MyDrive/submission_1159_finale.json
Counts by category: {3: 234, 1: 394, 2: 332, 4: 255}


## FINAL TRY 4

In [ ]:
# build_submission_robust.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt # Import from scipy.signal

ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT /"20251103_PS12"
OUT_JSON = ROOT / "submission_2_1104_finale.json"

SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Posterior thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Energy gate for vessel fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5            # median + Z*mad threshold

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def median_filter_probs(P, k=5):
    try:
        from scipy.ndimage import median_filter
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    # bandpass energy -> per-second RMS
    # use scipy.signal.iirfilter and scipy.signal.sosfiltfilt
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                            btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    # frame energy
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    # robust threshold
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    # map to seconds
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    # 60s window: ensure at least 1 vessel second per minute if energy present
    for s in range(0, total_sec, 60):
        seg = sec_flags[s:s+60]
        if seg.sum() == 0 and hot.sum() > 0:
            # pick the second with highest rms in that minute
            # approximate mapping: take frame index near center of window
            idx = min(len(rms)-1, int(np.rint(((s+30)*SR)/ENERGY_HOP)))
            sec_pick = min(total_sec-1, s + 30)
            sec_flags[sec_pick] = 1
    return sec_flags

def per_second_fusion(P, y, clip_duration):
    # P: [T,4] posterior; y: waveform
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None, 0]
    # aggregate max per second
    sec_scores = np.full((total_sec, 4), -np.inf, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])
    # voters
    voters = []
    # A: argmax
    voters.append(np.argmax(np.where(np.isfinite(sec_scores), sec_scores, -1e9), axis=1))
    # B: threshold masks
    thr = compute_adaptive_thresholds(P)
    masks = (np.where(np.isfinite(sec_scores), sec_scores, -1e9) >= thr.reshape(1,4)).astype(np.int32)
    # choose highest class among those passing, else -1
    b_choice = np.where(masks.sum(1)>0, np.argmax(sec_scores, axis=1), -1)
    voters.append(b_choice)
    # C: energy vessel fallback
    vessel_secs = energy_vessel_seconds(y, total_sec)
    c_choice = np.where(vessel_secs==1, 0, -1)  # 0-based vessel
    voters.append(c_choice)

    # majority fusion per second
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = [v[s] for v in voters if v[s] != -1]
        if not votes:
            # fallback to global argmax row if all -1
            row = sec_scores[s]
            if np.all(np.isneginf(row)):
                sec_cls[s] = None
                continue
            best = int(np.argmax(row))
            sec_cls[s] = best + 1
            continue
        # vote tally
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        if counts[winner] >= 2:
            sec_cls[s] = winner + 1
        else:
            # tie-break by posterior
            best = int(np.argmax(sec_scores[s]))
            sec_cls[s] = best + 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        P = torch.sigmoid(model(x)[0]).cpu().numpy().astype(np.float32)
    sec_cls, sec_scores = per_second_fusion(P, y, wav_duration)
    # to segments
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = sec_scores[s, c-1] if np.isfinite(sec_scores[s, c-1]) else 0.0
        s += 1
        while s < total_sec and sec_cls[s] == c:
            if np.isfinite(sec_scores[s, c-1]):
                max_sc = max(max_sc, sec_scores[s, c-1])
            s += 1
        et = s
        segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    return segs

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()

Wrote submission with 821 annotations to /content/drive/MyDrive/submission_2_1104_finale.json
Counts by category: {1: 427, 2: 361, 3: 27, 4: 6}


## FINAL TRY 5

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# Paths
ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "20251103_PS12"  # update if needed
OUT_JSON = ROOT / "submission_2_1116_finale.json"

# Audio/feature
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# Post-proc and calibration
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (tightened non-vessel)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004  # was 0.003

# De-chatter (stronger)
REASSIGN_MARGIN = 0.03    # was 0.02

# Vessel continuity (wider gap fill)
GAP_FILL_SEC = 12         # was 10
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # de-chatter (stronger margin)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # vessel continuity (wider gap fill and edges)
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1])
        s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1217 annotations to /content/drive/MyDrive/submission_2_1116_finale.json
Counts by category: {1: 452, 2: 331, 3: 279, 4: 155}


## FINAL TRY 6

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# ----------------------------
# Paths
# ----------------------------
ROOT = Path("/content/drive/MyDrive")
WAV_DIR = ROOT / "20251103_PS12"  # set to your target mock dataset folder
MODEL_PATH = ROOT / "mixes/models/crnn_sed_finetuned_both_smallwin.pt"
OUT_JSON = ROOT / "submission_2_1120_finale.json"

# ----------------------------
# Audio/feature
# ----------------------------
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# ----------------------------
# Post-proc and calibration
# ----------------------------
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Adaptive thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (strict)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004

# De-chatter
REASSIGN_MARGIN = 0.03  # stronger

# Vessel continuity
GAP_FILL_SEC = 12
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

# ----------------------------
# Model
# ----------------------------
class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            self._feat_per_step = C*Mp
        self.rnn = nn.GRU(input_size=self._feat_per_step, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x)
        B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

# ----------------------------
# Calibration
# ----------------------------
def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

# ----------------------------
# Energy fallback
# ----------------------------
def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

# ----------------------------
# Per-second fusion and stitching
# ----------------------------
def frames_to_seconds(i): return i * (HOP / SR)

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # De-chatter (remove isolated 1s flips)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # Vessel continuity: fill small gaps and edge extend
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

# ----------------------------
# Inference for one file
# ----------------------------
def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def run_file(model, wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    # per-audio calibration
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    # build segments + prune
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1]); s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class defensively
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

# ----------------------------
# IO and validation
# ----------------------------
def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

# ----------------------------
# Main
# ----------------------------
def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    model = CRNN_SED()
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    model.eval()

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(model, a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1247 annotations to /content/drive/MyDrive/submission_2_1120_finale.json
Counts by category: {3: 223, 4: 300, 2: 336, 1: 388}


## **NEW APPROACH WITH GROUND TRUTH**

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import librosa

# ----------------------------
# Paths and config
# ----------------------------
ROOT = Path("/content/drive/MyDrive/mixes")
OUT_DIR = ROOT / "models"; OUT_DIR.mkdir(parents=True, exist_ok=True)

# Update these to your two datasets and their GT JSONs
DATA_DIRS = [
    Path("/content/drive/MyDrive/2025_09_12_PS_12"),      # Mock-1 WAV folder
    Path("/content/drive/MyDrive/20251006_PS12"),      # Mock-2 WAV folder (I3_Test_XXX.wav)
]
GT_JSONS = [
    Path("/content/drive/MyDrive/gt_annotations.json"),  # Mock-1 GT-like JSON
    Path("/content/drive/MyDrive/20251007_I3_gt_annotations.json"),                    # Mock-2 GT JSON
]


PREV_MODEL_PATH = OUT_DIR / "crnn_sed.pt"                 # warm start if exists
SAVE_PATH = OUT_DIR / "crnn_sed_finetuned_both_smallwin.pt"  # new model file

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Audio/feature
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000
SPF = HOP / SR  # seconds per frame

# Training
BATCH = 2               # small batch for CPU
EPOCHS = 20
LR = 8e-4
WEIGHT_DECAY = 1e-4
N_CLASSES = 4
VAL_SPLIT = 0.2

# Windowing to limit RAM
TRAIN_WIN_SEC = 60      # random crop length (sec)
VAL_WIN_SEC = 60        # eval chunk length (sec)
STEP_SEC = 60           # non-overlap windows for val accumulation

# Loss and imbalance
USE_FOCAL = True
FOCAL_GAMMA = 1.5
CLASS_WEIGHTS = torch.tensor([1.0, 2.0, 1.5, 2.0], dtype=torch.float32)

# Augmentations (cheap)
AUG_GAIN_DB = 6.0
AUG_PINK_SNR_DB = 25.0

# Val thresholds (soft early, normal later)
VAL_THR_BASE = np.array([0.3, 0.2, 0.2, 0.15], dtype=np.float32)
SOFT_EPOCHS = 5  # use 0.5x thresholds for first few epochs
WARMUP_BCE_EPOCHS = 4  # use BCE before focal loss

# ----------------------------
# Audio + features
# ----------------------------
def load_wav(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return y

def logmel(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    return librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)

def waveform_gain(y, max_db=AUG_GAIN_DB):
    if max_db is None or max_db <= 0: return y
    g = np.random.uniform(-max_db, max_db)
    return y * (10.0 ** (g / 20.0))

def pink_noise(n):
    nrows = 16
    arr = np.random.randn(nrows, n)
    arr = np.cumsum(arr, axis=1)
    weights = 2.0 ** (-np.arange(nrows))
    y = (weights[:, None] * arr).sum(axis=0)
    y = y / (np.max(np.abs(y)) + 1e-9)
    return y.astype(np.float32)

def mix_pink_noise(y, snr_db=AUG_PINK_SNR_DB):
    if snr_db is None: return y
    n = len(y)
    p = pink_noise(n)
    rms_y = np.sqrt(np.mean(y**2) + 1e-12)
    rms_p = np.sqrt(np.mean(p**2) + 1e-12)
    target_rms_p = rms_y / (10.0 ** (snr_db / 20.0))
    p = p * (target_rms_p / (rms_p + 1e-12))
    out = y + p
    mx = np.max(np.abs(out)) + 1e-9
    if mx > 1.0: out = out / mx
    return out.astype(np.float32)

def to_onehot_per_frame(sec_labels, T_frames):
    T_sec = len(sec_labels)
    frame_secs = np.clip(np.rint(np.arange(T_frames) * SPF).astype(int), 0, max(T_sec - 1,0))
    Y = np.zeros((T_frames, 4), dtype=np.float32)
    for i in range(T_frames):
        lab = int(sec_labels[frame_secs[i]]) if T_sec>0 else 0
        if 1 <= lab <= 4: Y[i, lab-1] = 1.0
    return Y

def crop_seconds(y, sec_labels, win_sec, start_sec=None):
    total_sec = int(np.floor(len(y)/SR))
    if total_sec <= 1:
        return y, sec_labels
    if start_sec is None:
        s = 0 if total_sec <= win_sec else np.random.randint(0, total_sec - win_sec + 1)
    else:
        s = max(0, min(start_sec, max(0, total_sec - win_sec)))
    e = min(total_sec, s + win_sec)
    y_start = int(s * SR); y_end = int(e * SR)
    y_win = y[y_start:y_end]
    labs = sec_labels[s:e] if len(sec_labels)>0 else np.zeros(e - s, dtype=np.int32)
    return y_win, labs

# ----------------------------
# GT parsing (COCO-like JSON)
# ----------------------------
def load_gt_intervals(gt_json_path):
    with open(gt_json_path, "r") as f:
        data = json.load(f)
    audios = {a["id"]: a for a in data["audios"]}
    annos = data["annotations"]
    filename_to_intervals = {}
    for a_id,a_info in audios.items():
        fname = a_info.get("file_name","")
        if not fname.lower().endswith(".wav"): fname = f"{fname}.wav"
        filename_to_intervals[fname] = []
    for ann in annos:
        a_id = ann["audio_id"]
        if a_id not in audios: continue
        fname = audios[a_id].get("file_name","")
        if not fname.lower().endswith(".wav"): fname = f"{fname}.wav"
        filename_to_intervals.setdefault(fname, []).append({
            "start": int(ann["start_time"]),
            "end": int(ann["end_time"]),
            "class": int(ann["category_id"]),
        })
    filename_to_duration = {}
    for a_id,a_info in audios.items():
        fname = a_info.get("file_name","")
        if not fname.lower().endswith(".wav"): fname = f"{fname}.wav"
        dur = a_info.get("duration", None)
        if dur is not None:
            filename_to_duration[fname] = int(np.floor(float(dur)))
    return filename_to_intervals, filename_to_duration

def build_records_from_dataset(data_dir, gt_json_path):
    intervals_map, dur_map = load_gt_intervals(gt_json_path)
    wavs = sorted([p for p in Path(data_dir).glob("*.wav")])
    records = []
    for p in wavs:
        fname = p.name
        if fname not in intervals_map:
            stem = p.stem
            if f"{stem}.wav" in intervals_map:
                fname = f"{stem}.wav"
            else:
                continue
        duration_sec = dur_map.get(fname, None)
        if duration_sec is None or duration_sec <= 0:
            y_tmp, _ = librosa.load(str(p), sr=SR, mono=True)
            duration_sec = int(np.floor(len(y_tmp)/SR))
        arr = np.zeros(duration_sec, dtype=np.int32)
        for seg in intervals_map[fname]:
            c = int(seg["class"]); st = max(0, int(seg["start"])); et = max(st, int(seg["end"]))
            et = min(et, duration_sec)
            if 1 <= c <= 4 and et > st: arr[st:et] = c
        records.append({"wav_path": str(p), "sec_labels": arr})
    return records

def build_combined_records(data_dirs, gt_jsons):
    recs = []
    for d,g in zip(data_dirs, gt_jsons):
        recs.extend(build_records_from_dataset(d,g))
    random.seed(42); random.shuffle(recs)
    return recs

# ----------------------------
# Dataset with positive-biased sampling
# ----------------------------
class WindowedSET(Dataset):
    def __init__(self, records, train=True):
        self.records = records
        self.train = train
        if self.train:
            self.pos_seconds = []
            for rec in self.records:
                labs = rec["sec_labels"]
                pos_idxs = np.where(labs > 0)[0]
                self.pos_seconds.append(pos_idxs if len(pos_idxs)>0 else np.array([], dtype=int))
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        rec = self.records[i]
        wav_path = rec["wav_path"]; sec_labels = rec["sec_labels"]
        y = load_wav(wav_path)
        total_sec = int(np.floor(len(y)/SR))
        if self.train:
            start_sec = None
            if np.random.rand() < 0.5 and self.pos_seconds[i].size > 0:
                center = int(np.random.choice(self.pos_seconds[i]))
                half = max(1, TRAIN_WIN_SEC//2)
                start_sec = max(0, min(center - np.random.randint(0, half+1), max(0, total_sec - TRAIN_WIN_SEC)))
            y, labs = crop_seconds(y, sec_labels, TRAIN_WIN_SEC, start_sec=start_sec)
            y = waveform_gain(y, AUG_GAIN_DB)
            y = mix_pink_noise(y, AUG_PINK_SNR_DB)
            M = logmel(y)
            Y = to_onehot_per_frame(labs, M.shape[1])
            return torch.from_numpy(M).unsqueeze(0), torch.from_numpy(Y)
        else:
            return wav_path, sec_labels, total_sec

def collate_train(batch):
    maxT = max(x.shape[-1] for x,y in batch)
    Xs, Ys = [], []
    for x,y in batch:
        T = x.shape[-1]
        if T < maxT:
            pad = maxT - T
            x = F.pad(x, (0,pad))
            y = F.pad(y, (0,0,0,pad))
        Xs.append(x); Ys.append(y)
    return torch.stack(Xs,0), torch.stack(Ys,0)

# ----------------------------
# Model
# ----------------------------
class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            self._feat_per_step = C*Mp
        self.rnn = nn.GRU(input_size=self._feat_per_step, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        z = self.cnn(x)
        B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

def align_time(logits, labels):
    Bt, Tt, C = logits.shape; Tl = labels.shape[1]
    if Tt > Tl: logits = logits[:, :Tl]
    elif Tt < Tl: logits = F.pad(logits, (0,0,0,Tl-Tt))
    return logits

# ----------------------------
# Losses
# ----------------------------
class BalancedFocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=1.5):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        with torch.no_grad():
            p = torch.sigmoid(logits)
        focal = ((1 - torch.abs(p - targets)) ** self.gamma) * bce
        if self.class_weights is not None:
            focal = focal * self.class_weights.view(1,1,-1)
        return focal.mean()

# ----------------------------
# Validation helpers
# ----------------------------
def eventize_seconds_from_probs(probs, thr_vec, total_sec):
    T, C = probs.shape
    sec_scores = np.zeros((total_sec, C), dtype=np.float32)
    for i in range(T):
        s = int(np.rint(i * SPF))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], probs[i])
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        k = int(np.argmax(sec_scores[s]))
        if sec_scores[s, k] >= float(thr_vec[k]):
            sec_cls[s] = k+1
    events = []
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None: s += 1; continue
        st = s; m = sec_scores[s, c-1]; s += 1
        while s < total_sec and sec_cls[s] == c:
            m = max(m, sec_scores[s, c-1]); s += 1
        events.append({"cat": c, "st": st, "et": s, "score": float(m)})
    return events

# ----------------------------
# Train loop
# ----------------------------
def train():
    # Build combined records
    records = []
    for d,g in zip(DATA_DIRS, GT_JSONS):
        records.extend(build_records_from_dataset(d, g))
    if len(records) == 0:
        raise RuntimeError("No labeled WAVs found. Check DATA_DIRS and GT_JSONS alignment.")
    random.seed(42); random.shuffle(records)

    # Split
    n_total = len(records)
    n_val = max(1, int(VAL_SPLIT * n_total))
    n_tr = n_total - n_val
    rng = torch.Generator().manual_seed(42)
    tr_subset, va_subset = random_split(records, [n_tr, n_val], generator=rng)
    tr_recs = [records[i] for i in tr_subset.indices]
    va_recs = [records[i] for i in va_subset.indices]

    # Quick class coverage log
    def class_coverage(rr):
        cnt = np.zeros(5, dtype=np.int64); tot = 0
        for r in rr:
            labs = r["sec_labels"]; tot += labs.size
            for k in range(5): cnt[k] += np.sum(labs==k)
        frac = cnt / max(1, tot)
        print({"label_seconds_frac": {k: round(float(frac[k]),4) for k in range(5)}})
    class_coverage(tr_recs)

    tr_dl = DataLoader(WindowedSET(tr_recs, train=True), batch_size=BATCH, shuffle=True,
                       num_workers=0, collate_fn=collate_train)
    va_dl = DataLoader(WindowedSET(va_recs, train=False), batch_size=1, shuffle=False, num_workers=0)

    model = CRNN_SED().to(DEVICE)
    if PREV_MODEL_PATH.exists():
        model.load_state_dict(torch.load(PREV_MODEL_PATH, map_location=DEVICE))

    crit_focal = BalancedFocalLoss(CLASS_WEIGHTS.to(DEVICE) if USE_FOCAL else None, gamma=FOCAL_GAMMA)
    crit_bce = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_metric = -1.0
    for ep in range(1, EPOCHS+1):
        # Train
        model.train(); tr_loss=0.0; ntr=0
        for x,y in tr_dl:
            x,y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)
            logits = align_time(logits, y)
            if ep <= WARMUP_BCE_EPOCHS:
                loss = crit_bce(logits, y)
            else:
                loss = crit_focal(logits, y)
            loss.backward(); opt.step()
            tr_loss += float(loss.item())*x.size(0); ntr += x.size(0)
        tr_loss /= max(ntr,1)

        # Validation by streaming 60s windows, accumulating per-second scores
        model.eval(); val_recall_num=0; val_recall_den=0
        val_thr_now = VAL_THR_BASE * (0.5 if ep <= SOFT_EPOCHS else 1.0)
        with torch.no_grad():
            for (wav_path, sec_labels, total_sec) in va_dl:
                wav_path = wav_path[0]
                sec_labels = sec_labels[0].numpy()
                total_sec = int(total_sec[0].item())
                y = load_wav(wav_path)
                sec_scores = np.zeros((total_sec, N_CLASSES), dtype=np.float32)
                # slide non-overlapping windows
                for s0 in range(0, max(1,total_sec), STEP_SEC):
                    y_win, _ = crop_seconds(y, sec_labels, VAL_WIN_SEC, start_sec=s0)
                    M = logmel(y_win)
                    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0).to(DEVICE)
                    logits = model(x)
                    probs = torch.sigmoid(logits[0]).cpu().numpy()
                    T = probs.shape[0]
                    for i in range(T):
                        sec = s0 + int(np.rint(i * SPF))
                        if 0 <= sec < total_sec:
                            sec_scores[sec] = np.maximum(sec_scores[sec], probs[i])

                # Pred events
                pred_events = []
                sec_cls = [None]*total_sec
                for s in range(total_sec):
                    k = int(np.argmax(sec_scores[s]))
                    if sec_scores[s, k] >= float(val_thr_now[k]): sec_cls[s] = k+1
                s = 0
                while s < total_sec:
                    c = sec_cls[s]
                    if c is None: s += 1; continue
                    st = s; m = sec_scores[s, c-1]; s += 1
                    while s < total_sec and sec_cls[s] == c:
                        m = max(m, sec_scores[s, c-1]); s += 1
                    pred_events.append({"cat": c, "st": st, "et": s, "score": float(m)})

                # GT events
                gt_events = []
                s = 0
                while s < total_sec:
                    lab = int(sec_labels[s])
                    if lab==0: s += 1; continue
                    st = s; s += 1
                    while s < total_sec and int(sec_labels[s]) == lab: s += 1
                    gt_events.append({"cat": lab, "st": st, "et": s})
                val_recall_den += len(gt_events)
                for ge in gt_events:
                    g_st,g_et,gc = ge["st"], ge["et"], ge["cat"]
                    matched=False
                    for pe in pred_events:
                        if pe["cat"] != gc: continue
                        inter = max(0, min(g_et, pe["et"]) - max(g_st, pe["st"]))
                        if inter >= 1: matched=True; break
                    if matched: val_recall_num += 1

        val_recall = (val_recall_num/val_recall_den) if val_recall_den>0 else 0.0
        print({"epoch":ep,"train_loss":round(tr_loss,4),"val_event_recall":round(val_recall,4)})

        if val_recall > best_metric:
            best_metric = val_recall
            torch.save({k: v.cpu() for k,v in model.state_dict().items()}, SAVE_PATH)

    if not SAVE_PATH.exists():
        torch.save(model.state_dict(), SAVE_PATH)
    print("Saved:", SAVE_PATH)

if __name__ == "__main__":
    train()


{'label_seconds_frac': {0: 0.3148, 1: 0.4657, 2: 0.1246, 3: 0.072, 4: 0.023}}
{'epoch': 1, 'train_loss': 0.6702, 'val_event_recall': 0.3514}
{'epoch': 2, 'train_loss': 0.6222, 'val_event_recall': 0.4324}
{'epoch': 3, 'train_loss': 0.6169, 'val_event_recall': 0.4054}
{'epoch': 4, 'train_loss': 0.6169, 'val_event_recall': 0.4324}
{'epoch': 5, 'train_loss': 0.3197, 'val_event_recall': 0.3514}
{'epoch': 6, 'train_loss': 0.3045, 'val_event_recall': 0.1892}
{'epoch': 7, 'train_loss': 0.3023, 'val_event_recall': 0.0811}
{'epoch': 8, 'train_loss': 0.3009, 'val_event_recall': 0.027}
{'epoch': 9, 'train_loss': 0.3004, 'val_event_recall': 0.0541}
{'epoch': 10, 'train_loss': 0.3001, 'val_event_recall': 0.1081}
{'epoch': 11, 'train_loss': 0.3003, 'val_event_recall': 0.1081}
{'epoch': 12, 'train_loss': 0.3, 'val_event_recall': 0.1351}
{'epoch': 13, 'train_loss': 0.2996, 'val_event_recall': 0.1892}
{'epoch': 14, 'train_loss': 0.2999, 'val_event_recall': 0.3243}
{'epoch': 15, 'train_loss': 0.2995, 'va

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# ----------------------------
# Paths
# ----------------------------
ROOT = Path("/content/drive/MyDrive")
WAV_DIR = ROOT / "20251006_PS12"  # set to your target mock dataset folder
MODEL_PATH = ROOT / "mixes/models/crnn_sed_finetuned_both_smallwin.pt"
OUT_JSON = ROOT / "submission_last_diw1.json"

# ----------------------------
# Audio/feature
# ----------------------------
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# ----------------------------
# Post-proc and calibration
# ----------------------------
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Adaptive thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (strict)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004

# De-chatter
REASSIGN_MARGIN = 0.03  # stronger

# Vessel continuity
GAP_FILL_SEC = 12
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

# ----------------------------
# Model
# ----------------------------
class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            self._feat_per_step = C*Mp
        self.rnn = nn.GRU(input_size=self._feat_per_step, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x)
        B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

# ----------------------------
# Calibration
# ----------------------------
def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

# ----------------------------
# Energy fallback
# ----------------------------
def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

# ----------------------------
# Per-second fusion and stitching
# ----------------------------
def frames_to_seconds(i): return i * (HOP / SR)

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # De-chatter (remove isolated 1s flips)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # Vessel continuity: fill small gaps and edge extend
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

# ----------------------------
# Inference for one file
# ----------------------------
def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def run_file(model, wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    # per-audio calibration
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    # build segments + prune
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1]); s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class defensively
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

# ----------------------------
# IO and validation
# ----------------------------
def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

# ----------------------------
# Main
# ----------------------------
def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    model = CRNN_SED()
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    model.eval()

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(model, a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 1153 annotations to /content/drive/MyDrive/submission_last_diw1.json
Counts by category: {4: 230, 1: 351, 2: 324, 3: 248}


In [ ]:
# build_submission_robust.py
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt # Import from scipy.signal

ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "20251027_PS12/20251027_PS12"
OUT_JSON = ROOT / "submission_docker_2.json"

SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

# Posterior thresholds (floors/ceilings)
BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Energy gate for vessel fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5            # median + Z*mad threshold

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def median_filter_probs(P, k=5):
    try:
        from scipy.ndimage import median_filter
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    # bandpass energy -> per-second RMS
    # use scipy.signal.iirfilter and scipy.signal.sosfiltfilt
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                            btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    # frame energy
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    # robust threshold
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    # map to seconds
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    # 60s window: ensure at least 1 vessel second per minute if energy present
    for s in range(0, total_sec, 60):
        seg = sec_flags[s:s+60]
        if seg.sum() == 0 and hot.sum() > 0:
            # pick the second with highest rms in that minute
            # approximate mapping: take frame index near center of window
            idx = min(len(rms)-1, int(np.rint(((s+30)*SR)/ENERGY_HOP)))
            sec_pick = min(total_sec-1, s + 30)
            sec_flags[sec_pick] = 1
    return sec_flags

def per_second_fusion(P, y, clip_duration):
    # P: [T,4] posterior; y: waveform
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None, 0]
    # aggregate max per second
    sec_scores = np.full((total_sec, 4), -np.inf, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])
    # voters
    voters = []
    # A: argmax
    voters.append(np.argmax(np.where(np.isfinite(sec_scores), sec_scores, -1e9), axis=1))
    # B: threshold masks
    thr = compute_adaptive_thresholds(P)
    masks = (np.where(np.isfinite(sec_scores), sec_scores, -1e9) >= thr.reshape(1,4)).astype(np.int32)
    # choose highest class among those passing, else -1
    b_choice = np.where(masks.sum(1)>0, np.argmax(sec_scores, axis=1), -1)
    voters.append(b_choice)
    # C: energy vessel fallback
    vessel_secs = energy_vessel_seconds(y, total_sec)
    c_choice = np.where(vessel_secs==1, 0, -1)  # 0-based vessel
    voters.append(c_choice)

    # majority fusion per second
    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = [v[s] for v in voters if v[s] != -1]
        if not votes:
            # fallback to global argmax row if all -1
            row = sec_scores[s]
            if np.all(np.isneginf(row)):
                sec_cls[s] = None
                continue
            best = int(np.argmax(row))
            sec_cls[s] = best + 1
            continue
        # vote tally
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        if counts[winner] >= 2:
            sec_cls[s] = winner + 1
        else:
            # tie-break by posterior
            best = int(np.argmax(sec_scores[s]))
            sec_cls[s] = best + 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    x = torch.from_numpy(M).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        P = torch.sigmoid(model(x)[0]).cpu().numpy().astype(np.float32)
    sec_cls, sec_scores = per_second_fusion(P, y, wav_duration)
    # to segments
    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = sec_scores[s, c-1] if np.isfinite(sec_scores[s, c-1]) else 0.0
        s += 1
        while s < total_sec and sec_cls[s] == c:
            if np.isfinite(sec_scores[s, c-1]):
                max_sc = max(max_sc, sec_scores[s, c-1])
            s += 1
        et = s
        segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})
    return segs

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()

Wrote submission with 660 annotations to /content/drive/MyDrive/submission_docker_2.json
Counts by category: {1: 355, 2: 282, 3: 21, 4: 2}


In [ ]:
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import librosa, soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import iirfilter, sosfiltfilt

# Paths
ROOT = Path("/content/drive/MyDrive")
MODEL_PATH = ROOT / "mixes/models/crnn_sed.pt"
WAV_DIR = ROOT / "test_data"  # update if needed
OUT_JSON = ROOT / "submission_docker_5.json"

# Audio/feature
SR = 48000
N_MELS = 64
N_FFT = 2048
HOP = 1024
FMIN = 20
FMAX = 20000

# Post-proc and calibration
USE_MEDIAN_SMOOTHING = True
MEDIAN_K = 5

BASE = np.array([0.03, 0.02, 0.02, 0.015], dtype=np.float32)
FLOOR = np.array([0.015, 0.008, 0.008, 0.008], dtype=np.float32)
CEIL  = np.array([0.12, 0.08, 0.08, 0.06], dtype=np.float32)

# Pruning thresholds (tightened non-vessel)
PRUNE_MIN_VESSEL = 0.005
PRUNE_MIN_OTHER  = 0.004  # was 0.003

# De-chatter (stronger)
REASSIGN_MARGIN = 0.03    # was 0.02

# Vessel continuity (wider gap fill)
GAP_FILL_SEC = 12         # was 10
VESSEL_STRONG = 0.20
EDGE_EXTEND_SEC = 3

# Energy fallback
ENERGY_WIN = 2048
ENERGY_HOP = 1024
ENERGY_BAND = (50, 3000)  # Hz
ENERGY_Z = 1.5

# Categories
CATEGORIES = [
    {"id": 1, "name": "vessel"},
    {"id": 2, "name": "marine_animal"},
    {"id": 3, "name": "natural_sound"},
    {"id": 4, "name": "other_anthropogenic"},
]

class CRNN_SED(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,2)),
        )
        with torch.no_grad():
            dummy = torch.zeros(1,1,n_mels,400)
            z = self.cnn(dummy); _,C,Mp,Tp = z.shape
            feat_dim = C*Mp
        self.rnn = nn.GRU(input_size=feat_dim, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, 4)
    def forward(self, x):
        z = self.cnn(x); B,C,Mp,Tp = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, Tp, C*Mp)
        z,_ = self.rnn(z)
        return self.head(z)

model = CRNN_SED()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

def extract_logmel_and_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0)
    logS = librosa.power_to_db(S + 1e-12, ref=1.0).astype(np.float32)
    return logS, y

def frames_to_seconds(i): return i * (HOP / SR)

def calibrate_logits_per_audio(logits_np):
    Ts = [0.7, 0.9, 1.0, 1.1, 1.3]
    best_T = 1.0; best_spread = -1.0
    for T in Ts:
        P = 1.0 / (1.0 + np.exp(-logits_np / T))
        q90 = np.quantile(P, 0.9); q80 = np.quantile(P, 0.8)
        spread = q90 - q80
        if spread > best_spread:
            best_spread = spread; best_T = T
    P = 1.0 / (1.0 + np.exp(-logits_np / best_T))
    for c in range(P.shape[1]):
        pc = P[:,c]
        lo, hi = np.quantile(pc, 0.05), np.quantile(pc, 0.95)
        if hi > lo + 1e-6:
            P[:,c] = 0.01 + 0.98 * (np.clip(pc, lo, hi) - lo) / (hi - lo)
    return P

def median_filter_probs(P, k=5):
    try:
        out = np.empty_like(P)
        for c in range(P.shape[1]):
            out[:, c] = median_filter(P[:, c], size=k, mode="nearest")
        return out
    except Exception:
        return P

def compute_adaptive_thresholds(P):
    thr = []
    for c in range(4):
        pc = np.asarray(P[:, c], dtype=np.float32)
        q75, q90 = np.quantile(pc, [0.75, 0.90])
        t = max(BASE[c], q75 + 0.3*(q90 - q75))
        thr.append(float(np.clip(t, FLOOR[c], CEIL[c])))
    return np.array(thr, dtype=np.float32)

def energy_vessel_seconds(y, total_sec):
    sos = iirfilter(4, [ENERGY_BAND[0]/(SR/2), ENERGY_BAND[1]/(SR/2)],
                    btype='band', ftype='butter', output='sos')
    yb = sosfiltfilt(sos, y)
    frames = librosa.util.frame(yb, frame_length=ENERGY_WIN, hop_length=ENERGY_HOP).T
    rms = np.sqrt(np.mean(frames**2, axis=1) + 1e-12)
    med = np.median(rms); mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + ENERGY_Z * mad
    hot = (rms >= thr).astype(np.int32)
    sec_flags = np.zeros(total_sec, dtype=np.int32)
    for i,flag in enumerate(hot):
        t = int(np.rint((i*ENERGY_HOP)/SR))
        if 0 <= t < total_sec and flag:
            sec_flags[t] = 1
    return sec_flags

def decide_seconds(P, y, clip_duration):
    if USE_MEDIAN_SMOOTHING:
        P = median_filter_probs(P, k=MEDIAN_K)
    T = P.shape[0]
    total_sec = int(np.floor(float(clip_duration)))
    if total_sec <= 0:
        return [None]*0, np.zeros((0,4), dtype=np.float32)

    sec_scores = np.full((total_sec, 4), 0.0, dtype=np.float32)
    for i in range(T):
        s = int(np.rint(frames_to_seconds(i)))
        if 0 <= s < total_sec:
            sec_scores[s] = np.maximum(sec_scores[s], P[i])

    thr = compute_adaptive_thresholds(P)
    mask_pass = (sec_scores >= thr.reshape(1,4)).astype(np.int32)
    mask_choice = np.where(mask_pass.sum(1) > 0, np.argmax(sec_scores, axis=1), -1)
    argmax_choice = np.argmax(sec_scores, axis=1)
    vessel_secs = energy_vessel_seconds(y, total_sec)
    energy_choice = np.where(vessel_secs==1, 0, -1)

    sec_cls = [None]*total_sec
    for s in range(total_sec):
        votes = []
        if mask_choice[s] != -1: votes.append(mask_choice[s])
        if energy_choice[s] != -1: votes.append(energy_choice[s])
        votes.append(argmax_choice[s])
        counts = np.bincount(votes, minlength=4)
        winner = int(np.argmax(counts))
        sec_cls[s] = winner + 1

    # de-chatter (stronger margin)
    for s in range(1, total_sec-1):
        c = sec_cls[s]
        if c is None: continue
        if sec_cls[s-1] == sec_cls[s+1] and sec_cls[s-1] != c:
            neighbor = sec_cls[s-1]
            if sec_scores[s-1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN and \
               sec_scores[s+1, neighbor-1] - sec_scores[s, c-1] >= REASSIGN_MARGIN:
                sec_cls[s] = neighbor

    # vessel continuity (wider gap fill and edges)
    def strong_vessel(idx): return sec_scores[idx,0] >= VESSEL_STRONG
    # collect vessel blocks
    vblocks = []
    i = 0
    while i < total_sec:
        if sec_cls[i] == 1:
            st = i; i += 1
            while i < total_sec and sec_cls[i] == 1: i += 1
            vblocks.append((st, i))
        else:
            i += 1
    for k in range(len(vblocks)-1):
        a_st, a_et = vblocks[k]; b_st, b_et = vblocks[k+1]
        gap = b_st - a_et
        if 0 < gap <= GAP_FILL_SEC:
            if (a_et-1 >= 0 and strong_vessel(a_et-1)) and (b_st < total_sec and strong_vessel(b_st)):
                for t in range(a_et, b_st):
                    cur = sec_cls[t]
                    if cur != 1:
                        if sec_scores[t,0] >= (PRUNE_MIN_VESSEL if cur is None else 0.5*sec_scores[t,cur-1]):
                            sec_cls[t] = 1
    # edge extend
    if total_sec > 0 and sec_scores[0,0] >= VESSEL_STRONG:
        for t in range(min(EDGE_EXTEND_SEC, total_sec)):
            if sec_cls[t] is None: sec_cls[t] = 1
    if total_sec > 0 and sec_scores[-1,0] >= VESSEL_STRONG:
        for off in range(1, EDGE_EXTEND_SEC+1):
            idx = total_sec - off
            if idx >= 0 and sec_cls[idx] is None: sec_cls[idx] = 1

    return sec_cls, sec_scores

def run_file(wav_path, wav_duration):
    M, y = extract_logmel_and_audio(wav_path)
    with torch.no_grad():
        logits = model(torch.from_numpy(M).unsqueeze(0).unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    P = calibrate_logits_per_audio(logits)
    sec_cls, sec_scores = decide_seconds(P, y, wav_duration)

    segs = []
    total_sec = len(sec_cls)
    s = 0
    while s < total_sec:
        c = sec_cls[s]
        if c is None:
            s += 1; continue
        st = s; max_sc = float(sec_scores[s, c-1])
        s += 1
        while s < total_sec and sec_cls[s] == c:
            max_sc = max(max_sc, float(sec_scores[s, c-1])); s += 1
        et = s
        if et - st >= 1:
            if c == 1 and max_sc < PRUNE_MIN_VESSEL: continue
            if c != 1 and max_sc < PRUNE_MIN_OTHER: continue
            segs.append({"cat": c, "start": st, "end": et, "score": float(np.clip(max_sc,0,1))})

    # Merge touching same-class
    merged = []
    for ev in segs:
        if not merged:
            merged.append(ev); continue
        last = merged[-1]
        if ev["cat"] == last["cat"] and ev["start"] <= last["end"]:
            last["end"] = max(last["end"], ev["end"])
            last["score"] = max(last["score"], ev["score"])
        else:
            merged.append(ev)
    return merged

def build_audio_list(wav_dir):
    wavs = sorted([p for p in Path(wav_dir).glob("*.wav")])
    audios = []
    for i, p in enumerate(wavs, start=1):
        info = sf.info(str(p))
        audios.append({"id": i, "file_name": p.name, "file_path": str(p), "duration": float(info.duration)})
    return audios

def validate_submission_dict(subm):
    assert "info" in subm and isinstance(subm["info"], dict)
    assert "audios" in subm and isinstance(subm["audios"], list)
    assert "categories" in subm and isinstance(subm["categories"], list)
    assert "annotations" in subm and isinstance(subm["annotations"], list)
    cat_ids = sorted([c["id"] for c in subm["categories"]]); assert cat_ids == [1,2,3,4]
    audio_ids = [a["id"] for a in subm["audios"]]; assert audio_ids == list(range(1,len(audio_ids)+1))
    prev = 0
    for ann in subm["annotations"]:
        assert set(ann.keys()) == {"id","audio_id","category_id","start_time","end_time","duration","score"}
        assert isinstance(ann["id"], int) and ann["id"] > prev; prev = ann["id"]
        st = int(ann["start_time"]); et = int(ann["end_time"]); dur = int(ann["duration"])
        assert et > st and dur == (et - st)
        sc = float(ann["score"]); assert 0.0 <= sc <= 1.0

def summarize_counts(subm):
    import collections
    cnt = collections.Counter()
    for ann in subm["annotations"]:
        cnt[ann["category_id"]] += 1
    return dict(cnt)

def main():
    audios = build_audio_list(WAV_DIR)
    submission = {
        "info": {"description": "Grand Challenge UDA", "version": "1.0", "year": 2025},
        "audios": audios,
        "categories": CATEGORIES,
        "annotations": []
    }

    annotations = []
    ann_id = 1
    for a in audios:
        evs = run_file(a["file_path"], a["duration"])
        for ev in evs:
            st, et = int(ev["start"]), int(ev["end"])
            if et <= st: continue
            annotations.append({
                "id": ann_id,
                "audio_id": a["id"],
                "category_id": int(ev["cat"]),
                "start_time": st,
                "end_time": et,
                "duration": et - st,
                "score": float(ev["score"]),
            })
            ann_id += 1

    annotations.sort(key=lambda r: (r["audio_id"], r["start_time"], r["category_id"]))
    submission["annotations"] = annotations
    validate_submission_dict(submission)
    OUT_JSON.write_text(json.dumps(submission, indent=2))
    print(f"Wrote submission with {len(annotations)} annotations to {OUT_JSON}")
    print("Counts by category:", summarize_counts(submission))

if __name__ == "__main__":
    main()


Wrote submission with 31 annotations to /content/drive/MyDrive/submission_docker_5.json
Counts by category: {1: 13, 2: 11, 4: 1, 3: 6}
